In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.8 MB/s eta 0:00:00


In [ ]:
from Bio import AlignIO
import pandas as pd

# Load MSA
msa_path = ".../kcnq_msa.fasta" # EDIT PATH, file: kcnq_msa.fasta
alignment = AlignIO.read(msa_path, "fasta")

print(f"Loaded alignment with {len(alignment)} sequences "
      f"and length {alignment.get_alignment_length()}")

# Identify KCNQ2 sequence
kcnq2_id = None
for record in alignment:
    if "KCNQ2" in record.id:
        kcnq2_id = record.id
        break

if kcnq2_id is None:
    raise ValueError("❌ KCNQ2 not found in alignment IDs")

print("KCNQ2 sequence ID:", kcnq2_id)

# Build pos → alignment column map
pos_to_col = {}
pos = 0

kcnq2_seq = next(rec for rec in alignment if rec.id == kcnq2_id).seq

for col_idx, aa in enumerate(kcnq2_seq):
    if aa != "-":
        pos += 1
        pos_to_col[pos] = col_idx

print(f"Mapped {pos} KCNQ2 positions to alignment columns")

# Compute kcnqcon (like Heyne)
results = []

for pos, col in pos_to_col.items():
    aa_ref = kcnq2_seq[col]
    count = 0

    for rec in alignment:
        aa = rec.seq[col]
        if aa == aa_ref and aa != "-":
            count += 1

    results.append({
        "pos": pos,
        "kcnqcon": count
    })

kcnqcon_df = pd.DataFrame(results)

# Sanity checks
print(kcnqcon_df.head())
print("Unique kcnqcon values:", sorted(kcnqcon_df["kcnqcon"].unique()))

Loaded alignment with 5 sequences and length 1120
KCNQ2 sequence ID: sp_O43526_KCNQ2_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_2_OS_Homo_sapiens_OX_9606_GN_KCNQ2_PE_1_SV_2
Mapped 872 KCNQ2 positions to alignment columns
   pos  kcnqcon
0    1        5
1    2        1
2    3        1
3    4        2
4    5        1
Unique kcnqcon values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [ ]:
import pandas as pd
import numpy as np

# choose the KCNQ2 record
kcnq2_rec = next(rec for rec in alignment if rec.id == kcnq2_id)
kcnq2_seq = str(kcnq2_rec.seq)
L = alignment.get_alignment_length()

# build column -> KCNQ2_pos map (pos counts only non-gaps)
col_to_pos = {}
pos = 0
for col in range(L):
    if kcnq2_seq[col] != "-":
        pos += 1
        col_to_pos[col] = pos
    else:
        col_to_pos[col] = np.nan  # KCNQ2 has a gap here

# compute "kcnqcon" per alignment column (including gaps)
rows = []
for col in range(L):
    aa_ref = kcnq2_seq[col]

    if aa_ref == "-":
        rows.append({
            "aln_col": col,
            "KCNQ2_pos": col_to_pos[col],
            "KCNQ2_AA": aa_ref,
            "kcnqcon": np.nan
        })
        continue

    count = 0
    for rec in alignment:
        aa = rec.seq[col]
        if aa != "-" and aa == aa_ref:
            count += 1

    rows.append({
        "aln_col": col,
        "KCNQ2_pos": col_to_pos[col],
        "KCNQ2_AA": aa_ref,
        "kcnqcon": int(count)
    })

kcnqcon_by_col = pd.DataFrame(rows)

# final output: per original KCNQ2 position (drop columns where KCNQ2 is gap)
kcnqcon_by_pos = (
    kcnqcon_by_col
    .dropna(subset=["KCNQ2_pos"])
    .copy()
)

# KCNQ2_pos currently float because of NaN; cast to int
kcnqcon_by_pos["KCNQ2_pos"] = kcnqcon_by_pos["KCNQ2_pos"].astype(int)

kcnqcon_df2 = kcnqcon_by_pos[["KCNQ2_pos", "kcnqcon", "aln_col"]].rename(columns={"KCNQ2_pos": "pos"})

print(kcnqcon_df2.head())
print("Unique kcnqcon values:", sorted(kcnqcon_df2["kcnqcon"].dropna().unique()))
print("Number of KCNQ2 positions:", kcnqcon_df2["pos"].nunique())

   pos  kcnqcon  aln_col
0    1      5.0        0
1    2      1.0        1
2    3      1.0        2
3    4      2.0        3
4    5      1.0        4
Unique kcnqcon values: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
Number of KCNQ2 positions: 872


In [ ]:
region = kcnqcon_by_col[(kcnqcon_by_col["aln_col"] >= 170) & (kcnqcon_by_col["aln_col"] <= 180)]
print(region[["aln_col", "KCNQ2_pos", "KCNQ2_AA", "kcnqcon"]])

     aln_col  KCNQ2_pos KCNQ2_AA  kcnqcon
170      170       83.0        N      4.0
171      171       84.0        V      3.0
172      172       85.0        L      5.0
173      173       86.0        E      5.0
174      174       87.0        R      5.0
175      175       88.0        P      5.0
176      176       89.0        R      4.0
177      177       90.0        G      5.0
178      178       91.0        W      5.0
179      179        NaN        -      NaN
180      180       92.0        A      4.0


In [ ]:
kcnqcon_df2["kcnqcon"] = kcnqcon_df2["kcnqcon"].astype(int)
print(kcnqcon_df2.head())
print(kcnqcon_df2["kcnqcon"].dtype)

   pos  kcnqcon  aln_col
0    1        5        0
1    2        1        1
2    3        1        2
3    4        2        3
4    5        1        4
int64


In [ ]:
pos_list = [
    1,106,107,113,114,126,127,130,141,144,144,144,153,175,178,185,
    192,194,196,198,201,201,203,205,207,213,213,214,239,247,253,
    254,254,256,258,265,265,265,265,267,268,268,268,268,269,272,
    274,274,276,277,277,281,284,285,285,287,290,290,294,294,295,
    296,301,304,306,309,313,315,317,318,325,333,335,337,339,341,
    344,352,353,355,358,359,362,386,489,541,541,546,546,547,553,
    553,554,556,556,558,560,563,567,581,581,588,592,637,764
]

rows = []

for p in pos_list:
    match = kcnqcon_df2[kcnqcon_df2["pos"] == p]

    if match.empty:
        rows.append({
            "pos": p,
            "kcnqcon": None,
            "aln_col": None
        })
    else:
        row = match.iloc[0]
        rows.append({
            "pos": p,
            "kcnqcon": int(row["kcnqcon"]),
            "aln_col": int(row["aln_col"])
        })

subset_df = pd.DataFrame(rows)

print(subset_df.head(20))

    pos  kcnqcon  aln_col
0     1        5        0
1   106        5      194
2   107        5      195
3   113        4      201
4   114        5      202
5   126        5      214
6   127        1      215
7   130        5      218
8   141        3      229
9   144        5      232
10  144        5      232
11  144        5      232
12  153        4      241
13  175        5      263
14  178        5      266
15  185        3      273
16  192        4      280
17  194        5      282
18  196        4      284
19  198        5      286
